In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import copy

def heat_equation(u, XT, alpha_train):
    d1 = torch.autograd.grad(
        outputs = u,
        inputs = XT,
        grad_outputs = torch.ones_like(u),
        create_graph = True
    )[0]

    u_t = d1[:, 0:1]
    u_x = d1[:, 1:2]

    d2 = torch.autograd.grad(
        outputs = u_x,
        inputs = XT,
        grad_outputs = torch.ones_like(u_x),
        create_graph = True
    )[0]

    u_xx = d2[:, 1:2]

    return u_t - alpha_train * u_xx

alpha_true = 0.1

def exact_solution(XT, alpha_true = 0.1):
    return torch.exp(-alpha_true *  np.pi**2 * XT[:, 0:1]) * torch.sin(np.pi * XT[:, 1:2])

torch.manual_seed(42)

n_data = 200
XT = torch.rand(n_data, 2, dtype=torch.float, requires_grad=True)

u_true = exact_solution(XT)

n_train = 2000
XT_collocation = torch.rand(n_train, 2, dtype=torch.float, requires_grad=True)

torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(2, 20),
    nn.Tanh(),
    nn.Linear(20, 20),
    nn.Tanh(),
    nn.Linear(20, 20),
    nn.Tanh(),
    nn.Linear(20, 20),
    nn.Tanh(),
    nn.Linear(20, 20),
    nn.Tanh(),
    nn.Linear(20, 20),
    nn.Tanh(),
    nn.Linear(20, 20),
    nn.Tanh(),
    nn.Linear(20, 20),
    nn.Tanh(),
    nn.Linear(20, 1)
)

def loss_fcn(model, XT_collocation, XT, alpha_train):
    model.train()

    u_pred1 = model(XT_collocation)
    residual = heat_equation(u_pred1, XT_collocation, alpha_train)
    physics_loss = torch.mean(residual**2)

    u_pred2 = model(XT)
    data_loss = torch.mean((u_pred2 - u_true)**2)

    total_loss = physics_loss + data_loss

    return physics_loss, data_loss, total_loss

alpha_train = nn.Parameter(torch.tensor([1.0], dtype=torch.float))
optimizer = torch.optim.Adam(list(model.parameters()) + [alpha_train], lr=1e-3)

physics_loss_list = []
data_loss_list = []
total_loss_list = []
alpha_list = []

start_epoch = 0
n_epochs = 5000
patience = 500

best_epoch = start_epoch

for i in range(start_epoch, start_epoch+n_epochs):
    physics_loss, data_loss, total_loss = loss_fcn(model, XT_collocation, XT, alpha_train)
    optimizer.zero_grad()
    total_loss.backward(retain_graph=True)
    optimizer.step()

    physics_loss_list.append(physics_loss.item())
    data_loss_list.append(data_loss.item())
    total_loss_list.append(total_loss.item())
    alpha_list.append(alpha_train.item())

    if i % 100 == 0:
        print(f'epoch:', i)

    if total_loss < total_loss_list[best_epoch]:
        best_epoch = i
        best_model = copy.deepcopy(model)
    elif best_epoch <= i - patience:
        print('Training finished early')
        break

print('Need more training')

epochs = np.linspace(1, len(total_loss_list), len(total_loss_list))

fig, ax = plt.subplots(1,1,figsize = (6,3),dpi = 150)
ax.plot(epochs, total_loss_list, label='Total loss')
ax.plot(epochs, physics_loss_list, label='Physics loss', alpha=0.4)
ax.plot(epochs, data_loss_list, label='Data loss', alpha = 0.4)
ax.set_xlabel('Epoch',fontsize = 16)
ax.set_ylabel('Loss',fontsize = 16)
ax.set_yscale('log')
ax.set_title('Loss during training',fontsize = 20)
ax.tick_params(labelsize=12, which='both',top=True, right = True, direction='in')
ax.grid(color='xkcd:dark blue',alpha = 0.2)
ax.legend(loc='upper right',fontsize = 12)
plt.show()

fig, ax = plt.subplots(1,1,figsize = (6,3),dpi = 150)
ax.plot(epochs, alpha_list, label='$\\alpha$')
ax.hlines(0.1, 1, epochs[-1], linestyles='--', colors='red', label='True $\\alpha$')
ax.set_xlabel('Epoch',fontsize = 16)
ax.set_ylabel('$\\alpha$',fontsize = 16) 
ax.set_title('Predicted $\\alpha$',fontsize = 20)
ax.tick_params(labelsize=12, which='both',top=True, right = True, direction='in')
ax.grid(color='xkcd:dark blue',alpha = 0.2)
ax.legend(loc='upper right',fontsize = 12)
plt.show()

X_test = torch.linspace(0, 1, 200).reshape(-1, 1)
T_test = torch.ones_like(X_test) * 0.5
XT_test = torch.cat([X_test, T_test], dim=1)

u_exact = exact_solution(XT_test).detach()

best_model.eval()
with torch.no_grad():
    u_pred = best_model(XT_test).detach()

fig, ax = plt.subplots(1,1,figsize = (6,3),dpi = 150)
ax.plot(X_test, u_pred, label='Predicted solution')
ax.plot(X_test, u_exact, label='Exact solution', linestyle='--')
ax.set_xlabel('X',fontsize = 16)
ax.set_ylabel('u',fontsize = 16)
ax.set_title('Solution to 1D heat equation at t$=0.5$',fontsize = 20)
ax.tick_params(labelsize=12, which='both',top=True, right = True, direction='in')
ax.grid(color='xkcd:dark blue',alpha = 0.2)
ax.legend(loc='upper right',fontsize = 12)
plt.show()